## Loading dataset

In [1]:
from google.colab import drive
drive.mount('/content/drive')
import pandas as pd
import numpy as np
import scipy.stats as stats
folder = '/content/drive/MyDrive/'

Mounted at /content/drive


In [2]:
df = pd.read_csv(folder + 'room_based_shelters.csv')
df['OCCUPANCY_DATE'] = pd.to_datetime(df['OCCUPANCY_DATE'], errors='coerce')

In [3]:
sorted(df['OCCUPANCY_DATE'].dt.year.unique())

[np.int32(2021), np.int32(2022), np.int32(2024), np.int32(2025)]

In [4]:
df.columns

Index(['_id', 'OCCUPANCY_DATE', 'SHELTER_ID', 'LOCATION_ID',
       'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE', 'SECTOR', 'PROGRAM_AREA',
       'SERVICE_USER_COUNT', 'CAPACITY_TYPE', 'CAPACITY_ACTUAL_ROOM',
       'CAPACITY_FUNDING_ROOM', 'OCCUPIED_ROOMS', 'UNOCCUPIED_ROOMS',
       'UNAVAILABLE_ROOMS', 'OCCUPANCY_RATE_ROOMS'],
      dtype='object')

## Setting up features + splitting

In [5]:
# get time features set up
df['year'] = df['OCCUPANCY_DATE'].dt.year
df['month'] = df['OCCUPANCY_DATE'].dt.month
df['dayofweek'] = df['OCCUPANCY_DATE'].dt.dayofweek

In [6]:
# build features + target
df['pressure'] = df['OCCUPIED_ROOMS'] / df['CAPACITY_ACTUAL_ROOM']

# high pressure will be 1 and not high pressure (low pressure/normal) will be 0
# high pressure -> how full the shelther is
df['high_pressure'] = (df['pressure'] > 0.98).astype(int)

# sort data in time orders per shelther
df = df.sort_values(['SHELTER_ID', 'OCCUPANCY_DATE'])

# create the future target (each shelter is treated differently)
# future pressure means the next 2 week's pressure status... we are doing lagged target forecasting
df['future_pressure'] = df.groupby('SHELTER_ID')['high_pressure'].shift(-30)

df_model = df.dropna(subset=['future_pressure']).copy()

In [7]:
# features and target
X = df_model[[
    'SHELTER_ID',
    'SERVICE_USER_COUNT',
    'CAPACITY_ACTUAL_ROOM',
    'CAPACITY_FUNDING_ROOM',
    'year',
    'month',
    'dayofweek'
]]

y = df_model['future_pressure']

In [8]:
# split the data
train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']


## Cross validation for learning rate

In [9]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import f1_score
from sklearn.ensemble import HistGradientBoostingClassifier
import numpy as np

In [10]:
learning_rates = [0.01, 0.03, 0.05, 0.1, 0.2, 0.3, 0.4]

In [11]:
tscv = TimeSeriesSplit(n_splits=5)

In [12]:
results = {}

for lr in learning_rates:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=lr,
            max_depth=6,
            max_iter=200,
            class_weight="balanced"
        )

        model.fit(X_tr, y_tr)
        preds = model.predict(X_val)

        score = f1_score(y_val, preds, pos_label=1)
        fold_scores.append(score)

    results[lr] = np.mean(fold_scores)

print(results)
print("Best learning rate:", max(results, key=results.get))

{0.01: np.float64(0.7354928855728451), 0.03: np.float64(0.7584196838033963), 0.05: np.float64(0.7636784901585696), 0.1: np.float64(0.7562962638699451), 0.2: np.float64(0.7514788365584881), 0.3: np.float64(0.7379623945287146), 0.4: np.float64(0.7283454161455297)}
Best learning rate: 0.05


## Cross validation for max depth

In [13]:
depths = [3, 4, 5, 6, 7, 8, 10]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for depth in depths:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):
        X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=depth,
            max_iter=200
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        fold_scores.append(f1_score(y_val, preds))

    results[depth] = np.mean(fold_scores)

best_depth = max(results, key=results.get)
print(best_depth)

4


## Cross validation for max iterations

In [14]:
# values to test
iters = [100, 200, 500, 1000]

tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_iter in iters:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=3,
            max_iter=n_iter
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_iter] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best max_iter:", best_iter)

{100: np.float64(0.7995823934299966), 200: np.float64(0.7966868866453238), 500: np.float64(0.7891588249489689), 1000: np.float64(0.7661767286344178)}
Best max_iter: 100


## Cross validation for min samples leaf

In [15]:
leaves = [10, 20, 50, 100]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_leaves in leaves:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=3,
            max_iter=100,
            min_samples_leaf=n_leaves,
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_leaves] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best min_samples_leaf:", best_iter)

{10: np.float64(0.7916720944351092), 20: np.float64(0.7994896705586092), 50: np.float64(0.7959712363571485), 100: np.float64(0.7978946750484645)}
Best min_samples_leaf: 20


## Cross validation for l2 reg

In [16]:
regs = [0, 0.01, 0.1, 1.0, 10.0]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_regs in regs:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=3,
            max_iter=100,
            min_samples_leaf=20,
            l2_regularization=n_regs
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_regs] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best l2_reg:", best_iter)

{0: np.float64(0.8023825838692533), 0.01: np.float64(0.8023253384804091), 0.1: np.float64(0.7991300316037444), 1.0: np.float64(0.7974747905156448), 10.0: np.float64(0.8054681114603465)}
Best l2_reg: 10.0


## Cross validation for max leaf nodes

In [17]:
nodes = [15, 31, 63, 127]
tscv = TimeSeriesSplit(n_splits=5)

results = {}

for n_nodes in nodes:
    fold_scores = []

    for train_idx, val_idx in tscv.split(X_train):

        X_tr = X_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]

        y_tr = y_train.iloc[train_idx]
        y_val = y_train.iloc[val_idx]

        model = HistGradientBoostingClassifier(
            learning_rate=0.2,
            max_depth=3,
            max_iter=100,
            min_samples_leaf=20,
            l2_regularization=0.01,
            max_leaf_nodes=n_nodes
        )

        model.fit(X_tr, y_tr)

        preds = model.predict(X_val)

        score = f1_score(y_val, preds)
        fold_scores.append(score)

    results[n_nodes] = np.mean(fold_scores)

print(results)

best_iter = max(results, key=results.get)

print("Best max_leaf_nodes:", best_iter)

{15: np.float64(0.8030357811677528), 31: np.float64(0.7996251927145464), 63: np.float64(0.8002318370936541), 127: np.float64(0.7980050158359951)}
Best max_leaf_nodes: 15


## Training the model + Results

In [18]:
# train the model
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import classification_report

train = df_model[df_model['OCCUPANCY_DATE'] < '2025-01-01']
test = df_model[df_model['OCCUPANCY_DATE'] >= '2025-01-01']

X_train = train[X.columns]
y_train = train['future_pressure']

X_test = test[X.columns]
y_test = test['future_pressure']

model = HistGradientBoostingClassifier(
    learning_rate=0.2,
    max_depth=3,
    max_iter=100,
    min_samples_leaf=20,
    l2_regularization=0.01,
    max_leaf_nodes=127,
    class_weight="balanced"
)

model.fit(X_train, y_train)

preds = model.predict(X_test)

print(classification_report(y_test, preds))

              precision    recall  f1-score   support

         0.0       0.41      0.37      0.38       965
         1.0       0.87      0.89      0.88      4520

    accuracy                           0.79      5485
   macro avg       0.64      0.63      0.63      5485
weighted avg       0.79      0.79      0.79      5485



In [19]:
# when the model says not high pressure it's right 41% of the time
  # recall: 0.37... it catches 37% of all true “normal” cases
  # performance of a classification model -> F1-score: 0.39 (overall very weak performacne for the not high pressure class)
  # model is not good at identifying “safe / low-pressure” situations

# when the model says high pressure it's right 87% of the time
  # recall: 0.89... it only catches 89% of all true high pressure cases
  # performance of a classification model -> F1-score: 0.88 ( strong performance)
  # the model is excellent at detecting shelter stress/overload

# OVERALL: this model is a very strong early warning system for shelter overload, NOT a balanced classifier for both normal and high pressure states
  # accuracy -> f1-score: 0.80 (our data is biased because we have a lot more samples of high pressure classes)
  # macro avg: taking the average performance of each class separately
  # weighted avg: takes the number of samples (support #) in each class into account

In [20]:
# most shelter day observations in the test period were already operating above the 90% pressure threshold
# 5326/5485 ~= 82% of our test data is high pressure.

## Dummy classifier

In [21]:
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

dummy = DummyClassifier(strategy='most_frequent')

dummy.fit(X_train, y_train)

dummy_preds = dummy.predict(X_test)

print(classification_report(y_test, dummy_preds))

              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00       965
         1.0       0.82      1.00      0.90      4520

    accuracy                           0.82      5485
   macro avg       0.41      0.50      0.45      5485
weighted avg       0.68      0.82      0.74      5485



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [22]:
# Because ~97% of room-based shelter observations were classified as high pressure, the dummy achieved 97%
# accuracy by predicting all observations as high pressure. The HistGradientBoostingClassifier produced only
# a modest improvement in macro F1-score (0.51 vs. 0.49), indicating that the severe class imbalance limited
# the model's ability to distinguish normal-pressure periods.

## Shelter pressure Ranking (30-day outlook)

In [23]:
# set up model probabilites
df_model['risk_prob'] = model.predict_proba(X)[:, 1]

In [24]:
# aggregate by shelter
  # avg_future_risk: overall long-term stress level
  # max_future_risk: captures spikes
  # high_risk_days: % of danger periods (0.7 -> 70% of days are high pressure)
  # total_days: how much data we have per shelter

shelter_risk = df_model.groupby('SHELTER_ID').agg(
    avg_future_risk=('risk_prob', 'mean'),
    max_future_risk=('risk_prob', 'max'),
    high_risk_days=('risk_prob', lambda x: (x > 0.8).mean()),
    total_days=('risk_prob', 'count')
).reset_index()

In [25]:
# rank the shelters by risk
top_risk_shelters = shelter_risk.sort_values(
    'avg_future_risk',
    ascending=False
)

In [26]:
# add location + sector info
locations = df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE', 'SECTOR']].drop_duplicates()

top_risk_shelters = top_risk_shelters.merge(
    locations,
    on='SHELTER_ID',
    how='left'
)

## Mapping shelter risks

In [27]:
!pip install geopy

In [28]:
df['FULL_ADDRESS'] = (df['LOCATION_ADDRESS'].astype(str) + ", Toronto, Canada")

In [29]:
from geopy.geocoders import Nominatim
import time

geolocator = Nominatim(user_agent="shelter_map")

coords = {}

unique_addresses = df[['SHELTER_ID', 'FULL_ADDRESS']].drop_duplicates()

for _, row in unique_addresses.iterrows():
    address = row['FULL_ADDRESS']
    if address in coords:
        continue
    try:
        location = geolocator.geocode(address)
        coords[address] = (location.latitude, location.longitude) if location else (None, None)
    except:
        coords[address] = (None, None)
    time.sleep(1)

In [30]:
coord_df = pd.DataFrame.from_dict(coords, orient='index', columns=['lat', 'lon'])
coord_df['FULL_ADDRESS'] = coord_df.index
coord_df.reset_index(drop=True, inplace=True)

In [31]:
top_risk_shelters['FULL_ADDRESS'] = top_risk_shelters['LOCATION_ADDRESS'].astype(str) + ", Toronto, Canada"
map_df = top_risk_shelters.merge(coord_df, on='FULL_ADDRESS', how='left')
map_df = map_df.dropna(subset=['lat', 'lon'])

In [32]:
import folium

toronto_map = folium.Map(location=[43.7, -79.4], zoom_start=11)

In [33]:
def color(r):
    if r > 0.85:
        return 'red'
    elif r > 0.7:
        return 'orange'
    else:
        return 'green'

In [34]:
for _, row in map_df.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=6,
        color=color(row['avg_future_risk']),
        fill=True,
        fill_opacity=0.7,
        popup=f"{row['LOCATION_ADDRESS']} | Risk: {row['avg_future_risk']:.2f}"
    ).add_to(toronto_map)

toronto_map

## Export coordinates + pressure risk level

In [37]:
def pressure_label(r):
    if r > 0.85:
        return 'High'
    elif r > 0.7:
        return 'Medium'
    else:
        return 'Low'

export_df = map_df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LOCATION_POSTAL_CODE',
                     'lat', 'lon', 'avg_future_risk']].copy()
export_df = export_df.drop_duplicates(subset=['SHELTER_ID', 'LOCATION_ADDRESS'])
export_df['pressure_risk'] = export_df['avg_future_risk'].apply(pressure_label)
export_df = export_df.rename(columns={'lat': 'LATITUDE', 'lon': 'LONGITUDE'})

export_df.to_csv(folder + 'room_based_shelters_map.csv', index=False)
print(f"Saved {len(export_df)} shelters")
print(export_df[['SHELTER_ID', 'LOCATION_ADDRESS', 'LATITUDE', 'LONGITUDE', 'pressure_risk']].to_string(index=False))

Saved 36 shelters
 SHELTER_ID        LOCATION_ADDRESS  LATITUDE  LONGITUDE pressure_risk
         24        45 The Esplanade 43.646654 -79.374119          High
         24      22 Metropolitan Rd 43.769531 -79.300911          High
         99 2387 Dundas Street West 43.657630 -79.453055          High
         40           640 Dixon Rd. 43.692159 -79.576585          High
         36         26 Gerrard St E 43.659567 -79.381006          High
         25              60 York St 43.646256 -79.383169          High
         83      808 Mt Pleasant Rd 43.708859 -79.390844          High
         98            970 Dixon Rd 43.687554 -79.600373          High
         98  5515 Eglinton Ave West 43.655382 -79.598607          High
         91         1684 Queen St E 43.667188 -79.313429          High
          2        4222 Kingston Rd 43.760736 -79.196700          High
          2        4674 Kingston Rd 43.776390 -79.174713          High
          2        4540 Kingston Rd 43.772250 -79.185890   

In [38]:
from google.colab import files
files.download(folder + 'room_based_shelters_map.csv')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>